# Smart Credit: Automating Financial Risk Brackets with Predictive Machine Learning Models

In [ ]:
import os, sys, re, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier,ExtraTreesClassifier,HistGradientBoostingClassifier,VotingClassifier)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import joblib  # for saving/loading the model

warnings.filterwarnings('ignore')
np.random.seed(42)
print('System initialized.')

System initialized.


In [ ]:
train_df = pd.read_csv('train.csv', low_memory=False)
test_df  = pd.read_csv('test.csv',  low_memory=False)
print(f'Train: {train_df.shape}  |  Test: {test_df.shape}')

Train: (100000, 28)  |  Test: (50000, 27)


In [8]:
def parse_credit_history_age(series):
    def _parse(val):
        if pd.isna(val):
            return np.nan
        m = re.search(r'(\d+)\s*Years?\s*(?:and)?\s*(\d+)\s*Months?', str(val))
        return (int(m.group(1)) * 12 + int(m.group(2))) if m else np.nan
    return series.apply(_parse)


In [ ]:
def clean_features_and_loans(df):
    """
    Cleans raw noisy values, clamps outliers, applies customer-level ffill/bfill,
    and creates binary indicator columns for loan types.
    """
    data = df.copy()

    # 1. Demographics & Occupation
    s = data['Age'].astype(str).str.replace('_', '', regex=False).str.strip()
    age_num = pd.to_numeric(s, errors='coerce')
    age_num.loc[(age_num < 14) | (age_num > 100)] = np.nan
    data['Age'] = age_num
    data['Age'] = data.groupby('Customer_ID', sort=False)['Age'].ffill().bfill()

    data['Occupation'] = data['Occupation'].replace(['_______', ''], np.nan)
    data['Occupation'] = data.groupby('Customer_ID', sort=False)['Occupation'].ffill().bfill()

    # 2. Income & Salary
    s = data['Annual_Income'].astype(str).str.replace('_', '', regex=False).str.strip()
    data['Annual_Income'] = pd.to_numeric(s, errors='coerce')
    data['Annual_Income'] = data.groupby('Customer_ID', sort=False)['Annual_Income'].ffill().bfill()

    data['Monthly_Inhand_Salary'] = data.groupby('Customer_ID', sort=False)['Monthly_Inhand_Salary'].ffill().bfill()
    data['Monthly_Inhand_Salary'] = data['Monthly_Inhand_Salary'].fillna(data['Annual_Income'] / 12.0)

    # 3. Banking & Accounts
    data.loc[(data['Num_Bank_Accounts'] < 0) | (data['Num_Bank_Accounts'] > 20), 'Num_Bank_Accounts'] = np.nan
    data['Num_Bank_Accounts'] = data.groupby('Customer_ID', sort=False)['Num_Bank_Accounts'].ffill().bfill()

    data.loc[(data['Num_Credit_Card'] < 0) | (data['Num_Credit_Card'] > 25), 'Num_Credit_Card'] = np.nan
    data['Num_Credit_Card'] = data.groupby('Customer_ID', sort=False)['Num_Credit_Card'].ffill().bfill()

    data.loc[(data['Interest_Rate'] < 1) | (data['Interest_Rate'] > 50), 'Interest_Rate'] = np.nan
    data['Interest_Rate'] = data.groupby('Customer_ID', sort=False)['Interest_Rate'].ffill().bfill()

    s = data['Num_of_Loan'].astype(str).str.replace('_', '', regex=False).str.strip()
    n_loan = pd.to_numeric(s, errors='coerce')
    n_loan.loc[(n_loan < 0) | (n_loan > 25)] = np.nan
    data['Num_of_Loan'] = n_loan
    data['Num_of_Loan'] = data.groupby('Customer_ID', sort=False)['Num_of_Loan'].ffill().bfill()

    # 4. Payment Delays & Inquiries
    data['Delay_from_due_date'] = np.maximum(0, data['Delay_from_due_date'])

    s = data['Num_of_Delayed_Payment'].astype(str).str.replace('_', '', regex=False).str.strip()
    n_del = pd.to_numeric(s, errors='coerce')
    n_del.loc[(n_del < 0) | (n_del > 50)] = np.nan
    data['Num_of_Delayed_Payment'] = n_del
    data['Num_of_Delayed_Payment'] = data.groupby('Customer_ID', sort=False)['Num_of_Delayed_Payment'].ffill().bfill()

    s = data['Changed_Credit_Limit'].astype(str).str.replace('_', '', regex=False).str.strip()
    data['Changed_Credit_Limit'] = pd.to_numeric(s, errors='coerce')
    data['Changed_Credit_Limit'] = data.groupby('Customer_ID', sort=False)['Changed_Credit_Limit'].ffill().bfill()

    data.loc[(data['Num_Credit_Inquiries'] < 0) | (data['Num_Credit_Inquiries'] > 30), 'Num_Credit_Inquiries'] = np.nan
    data['Num_Credit_Inquiries'] = data.groupby('Customer_ID', sort=False)['Num_Credit_Inquiries'].ffill().bfill()

    # 5. Debt, Credit History & Monthly Metrics
    data['Credit_Mix'] = data['Credit_Mix'].replace(['_', ''], np.nan)
    data['Credit_Mix'] = data.groupby('Customer_ID', sort=False)['Credit_Mix'].ffill().bfill()

    s = data['Outstanding_Debt'].astype(str).str.replace('_', '', regex=False).str.strip()
    data['Outstanding_Debt'] = pd.to_numeric(s, errors='coerce')
    data['Outstanding_Debt'] = data.groupby('Customer_ID', sort=False)['Outstanding_Debt'].ffill().bfill()

    data['Credit_History_Months'] = parse_credit_history_age(data['Credit_History_Age'])
    data['Credit_History_Months'] = data.groupby('Customer_ID', sort=False)['Credit_History_Months'].ffill().bfill()

    data['Payment_of_Min_Amount'] = data['Payment_of_Min_Amount'].map({'No': 0, 'NM': 0, 'Yes': 1}).fillna(0).astype(int)

    data.loc[data['Total_EMI_per_month'] > 2000, 'Total_EMI_per_month'] = np.nan
    data['Total_EMI_per_month'] = data.groupby('Customer_ID', sort=False)['Total_EMI_per_month'].ffill().bfill()

    s = data['Amount_invested_monthly'].astype(str).str.replace('_', '', regex=False).str.strip()
    inv_num = pd.to_numeric(s, errors='coerce')
    inv_num.loc[inv_num >= 10000] = np.nan
    data['Amount_invested_monthly'] = inv_num
    data['Amount_invested_monthly'] = data.groupby('Customer_ID', sort=False)['Amount_invested_monthly'].ffill().bfill()

    data['Payment_Behaviour'] = data['Payment_Behaviour'].replace(['!@9#%8', ''], np.nan)
    data['Payment_Behaviour'] = data.groupby('Customer_ID', sort=False)['Payment_Behaviour'].ffill().bfill()

    s = data['Monthly_Balance'].astype(str).str.replace('_', '', regex=False).str.strip()
    bal_num = pd.to_numeric(s, errors='coerce')
    bal_num.loc[bal_num < 0] = np.nan
    data['Monthly_Balance'] = bal_num
    data['Monthly_Balance'] = data.groupby('Customer_ID', sort=False)['Monthly_Balance'].ffill().bfill()

    # 6. Type of Loan Extraction
    loan_types = ['Auto Loan', 'Credit-Builder Loan', 'Personal Loan', 'Home Equity Loan',
                  'Mortgage Loan', 'Student Loan', 'Debt Consolidation Loan', 'Payday Loan']
    loan_str = data['Type_of_Loan'].fillna('')
    for lt in loan_types:
        col_name = 'Loan_' + lt.replace(' ', '_').replace('-', '_')
        data[col_name] = loan_str.str.contains(lt, regex=False).astype(int)

    return data

In [14]:
def clean_credit_data(df, is_train=True, learned_stats=None):
    """
    Complete cleaning pipeline wrapping feature cleaning and fallback imputation.
    """
    data = clean_features_and_loans(df)

    if is_train:
        learned_stats = {
            'Age': data['Age'].median(),
            'Occupation': 'Other',
            'Annual_Income': data['Annual_Income'].median(),
            'Monthly_Inhand_Salary': data['Monthly_Inhand_Salary'].median(),
            'Num_Bank_Accounts': data['Num_Bank_Accounts'].median(),
            'Num_Credit_Card': data['Num_Credit_Card'].median(),
            'Interest_Rate': data['Interest_Rate'].median(),
            'Num_of_Loan': data['Num_of_Loan'].median(),
            'Num_of_Delayed_Payment': data['Num_of_Delayed_Payment'].median(),
            'Changed_Credit_Limit': data['Changed_Credit_Limit'].median(),
            'Num_Credit_Inquiries': data['Num_Credit_Inquiries'].median(),
            'Credit_Mix': 'Standard',
            'Outstanding_Debt': data['Outstanding_Debt'].median(),
            'Credit_History_Months': data['Credit_History_Months'].median(),
            'Total_EMI_per_month': data['Total_EMI_per_month'].median(),
            'Amount_invested_monthly': data['Amount_invested_monthly'].median(),
            'Payment_Behaviour': 'Low_spent_Small_value_payments',
            'Monthly_Balance': data['Monthly_Balance'].median()
        }

    # Fill any remaining NaNs using learned stats
    for col, val in learned_stats.items():
        if col in data.columns:
            data[col] = data[col].fillna(val)

    data['Credit_Mix_Score'] = data['Credit_Mix'].map({'Bad': 0, 'Standard': 1, 'Good': 2}).fillna(1).astype(int)
    
    return data, learned_stats

print('Cleaning pipeline compiled.')

Cleaning pipeline compiled.


In [ ]:
train_clean, learned_stats = clean_credit_data(train_df, is_train=True)

print("Data Cleaning Complete!")
print(f"Train Shape: {train_clean.shape}")
print(f"Remaining Missing Values in Train: {train_clean.isnull().sum().sum()}")

# Optional: Clean test data using learned stats from training set
if 'test_df' in globals():
    test_clean, _ = clean_credit_data(test_df, is_train=False, learned_stats=learned_stats)
    print(f"Test Shape: {test_clean.shape}")
    print(f"Remaining Missing Values in Test: {test_clean.isnull().sum().sum()}")

Data Cleaning Complete!
Train Shape: (100000, 38)
Remaining Missing Values in Train: 30423
Test Shape: (50000, 37)
Remaining Missing Values in Test: 15189


In [ ]:
def engineer_financial_features(df):
    data = df.copy()
    data['Debt_to_Income']        = data['Outstanding_Debt']        / (data['Annual_Income']          + 1.0)
    data['EMI_to_Salary']         = data['Total_EMI_per_month']     / (data['Monthly_Inhand_Salary']  + 1.0)
    data['Invest_to_Salary']      = data['Amount_invested_monthly'] / (data['Monthly_Inhand_Salary']  + 1.0)
    data['Savings_Ratio']         = data['Monthly_Balance']         / (data['Monthly_Inhand_Salary']  + 1.0)
    data['Card_to_Account_Ratio'] = data['Num_Credit_Card']         / (data['Num_Bank_Accounts']      + 1.0)
    data['Delay_Ratio']           = data['Num_of_Delayed_Payment']  / ((data['Credit_History_Months'] / 12.0) + 1.0)
    return data

train_clean, learned_stats = clean_credit_data(train_df, is_train=True)
train_clean = engineer_financial_features(train_clean)
print(f'Features after engineering: {train_clean.shape[1]}')

Features after engineering: 44


In [ ]:
target_mapping     = {'Poor': 0, 'Standard': 1, 'Good': 2}
inv_target_mapping = {v: k for k, v in target_mapping.items()}

categorical_cols = ['Occupation', 'Credit_Mix', 'Payment_Behaviour']
train_encoded    = pd.get_dummies(train_clean, columns=categorical_cols, drop_first=True)

drop_cols    = ['ID', 'Customer_ID', 'Name', 'SSN', 'Month',
                'Type_of_Loan', 'Credit_History_Age', 'Credit_Score']
feature_cols = [c for c in train_encoded.columns if c not in drop_cols]

X = train_encoded[feature_cols].fillna(0)
y = train_clean['Credit_Score'].map(target_mapping)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {X_train.shape}  Val: {X_val.shape}')

Train: (80000, 54)  Val: (20000, 54)


In [ ]:
# Model Training 
rf_clf = RandomForestClassifier(
    n_estimators=200, max_depth=20, min_samples_leaf=5, n_jobs=-1, random_state=42
)
rf_clf.fit(X_train, y_train)
print('Random Forest trained.')

hgb_clf = HistGradientBoostingClassifier(
    max_iter=300, max_depth=8, learning_rate=0.05, random_state=42
)
hgb_clf.fit(X_train, y_train)
print('HistGradientBoosting trained.')

# Ensemble evaluation
rf_probs        = rf_clf.predict_proba(X_val)
hgb_probs       = hgb_clf.predict_proba(X_val)
ensemble_probs  = 0.60 * rf_probs + 0.40 * hgb_probs
ensemble_preds  = np.argmax(ensemble_probs, axis=1)
ens_acc         = accuracy_score(y_val, ensemble_preds)
print(f'\nEnsemble Validation Accuracy: {ens_acc*100:.2f}%')

Random Forest trained.
HistGradientBoosting trained.

Ensemble Validation Accuracy: 79.14%


In [ ]:
model_bundle = {
    'rf':                 rf_clf,
    'hgb':                hgb_clf,
    'feature_cols':       feature_cols,
    'categorical_cols':   categorical_cols,
    'learned_stats':      learned_stats,
    'target_mapping':     target_mapping,
    'inv_target_mapping': inv_target_mapping,
}

joblib.dump(model_bundle, 'credit_model.pkl')
print('✅  credit_model.pkl saved successfully!')
print(f'   Feature count : {len(feature_cols)}')
print(f'   Ensemble val  : {ens_acc*100:.2f}%')

✅  credit_model.pkl saved successfully!
   Feature count : 54
   Ensemble val  : 79.14%
